<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/torch/basic/custom-dataset/custom-dataset.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("custom-dataset", ...)`


# Problem: Write a custom Dataset and Dataloader to load from a CSV file

### Problem Statement
You are tasked with creating a **custom Dataset** and **Dataloader** in PyTorch to load data from a `data.csv` file (columns `X` and `y`, generated below). The loaded data will be used to train a pre-implemented linear regression model.

### Requirements
1. **Dataset Class**:
   - Implement a class `LinearRegressionDataset(csv_file)` that subclasses `torch.utils.data.Dataset` and:
     - Reads the CSV **once**, in `__init__`, and stores the features (`X`) and targets (`y`) separately as `float32` tensors.
     - Implements `__len__` (number of data rows, not counting the header) and `__getitem__(idx)`, which returns the pair `(x, y)` for row `idx`, in file order.
     - Each `x` and `y` has shape `(1,)`, so a batch of `B` items is `(B, 1)` — what `nn.Linear(1, 1)` and `nn.MSELoss` expect. (A target batch of shape `(B,)` against predictions of shape `(B, 1)` would silently broadcast to `(B, B)` inside the loss.)

2. **Dataloader**:
   - Use PyTorch's `DataLoader` to create an iterable for batch loading the dataset.
   - Use a batch size of 32 and shuffle the data every epoch.

Do not batch or shuffle inside the `Dataset` — that is the `DataLoader`'s job.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd

In [ ]:
# TODO: Implement the Dataset
class LinearRegressionDataset(Dataset):
    ...

In [ ]:
# Generate synthetic data and save it to data.csv
torch.manual_seed(42)
X = torch.rand(100, 1) * 10  # 100 data points between 0 and 10
y = 2 * X + 3 + torch.randn(100, 1)  # Linear relationship with noise

data = torch.cat((X, y), dim=1)
df = pd.DataFrame(data.numpy(), columns=['X', 'y'])
df.to_csv('data.csv', index=False)

In [ ]:
# Example usage of the DataLoader
dataset = LinearRegressionDataset('data.csv')
# TODO: Create a DataLoader with batch_size=32 that shuffles the data
dataloader = ...

batch_X, batch_y = next(iter(dataloader))
print(batch_X.shape, batch_y.shape)  # expect torch.Size([32, 1]) torch.Size([32, 1])

In [ ]:
# Define the Linear Regression Model
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)  # Single input and single output

    def forward(self, x):
        return self.linear(x)

# Initialize the model, loss function, and optimizer
model = LinearRegressionModel()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Training loop
epochs = 1000
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_X, batch_y in dataloader:
        # Forward pass
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Weight by batch size: the last batch is smaller (100 = 32 + 32 + 32 + 4)
        epoch_loss += loss.item() * len(batch_X)

    # Log the mean loss over the epoch every 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {epoch_loss / len(dataset):.4f}")

In [ ]:
# Display the learned parameters
[w, b] = model.linear.parameters()
print(f"Learned weight: {w.item():.4f}, Learned bias: {b.item():.4f}")

# Testing on new data
X_test = torch.tensor([[4.0], [7.0]])
with torch.no_grad():
    predictions = model(X_test)
    print(f"Predictions for {X_test.tolist()}: {predictions.tolist()}")